# Codeword optimization for the imbalance code

This notebook finds the best encoding of a logical qubit in the confined long-range chain. An encoding is a choice of

1. the excitation sector $b$,
2. the **pair assignment** $t$: which mirror pairs are empty, active, or doubly occupied,
3. the **logical pair** $(S,T)$: which two codewords become $|0_L\rangle$ and $|1_L\rangle$.

Every encoding passes the same symmetry checks. They differ in how strongly the Hamiltonian protects them, and that is what we compute.

**Plan of the notebook**

| § | What | Check |
|---|---|---|
| 1 | $V$, $W$ and configuration energies $E(c)$ | matches QuTiP $H_{\rm eff}$ exactly |
| 2 | The codespace as a list of codewords | $\dim\mathcal C=\tfrac12\binom{2K}{K}$ |
| 3 | Hop gaps and leakage $L(c)$ | worked example |
| 4 | Pair quantities $d_X$, $h(S,T)$, $\Delta E_{ST}$ | worked example |
| 5 | Exhaustive optimization | ranked table |
| 6 | Exact verification by ED | leakage formula vs. exact dynamics |
| 7 | Sweep over $\alpha$ | best encoding vs. $\alpha$ |

**Conventions**

- Sites $i=0,\dots,N-1$. The mirror of site $i$ is $N-1-i$.
- Mirror pair $p$ is the sites $(p,\;N-1-p)$, for $p=0,\dots,N/2-1$.
- Excitation = occupied site, $\hat n_i=|1\rangle\langle1|$ (QuTiP's `basis(2,1)`).

**Not included here:** sector leakage ($b\to b\pm1,b\pm2$), which depends only on $(N,\alpha,b)$ and enters the outer loop over $b$; and open-system noise.

In [1]:
import itertools
from math import comb

import numpy as np
import qutip as qt

## 1. The Hamiltonian inside a sector

For $J_z\gg J_x,B_x$, sector confinement removes particle-number-changing processes, and inside sector $b$ the dynamics is

$$H^b_{\rm eff}=T+U,\qquad T=J_x\sum_{i=0}^{N-2}\big(\hat a_i^\dagger\hat a_{i+1}+{\rm h.c.}\big),\qquad U=\sum_iV(i)\,\hat n_i+\sum_{i<j}W(i,j)\,\hat n_i\hat n_j,$$

with

$$V(i)=-2J_z\sum_{j\ne i}|i-j|^{-\alpha},\qquad W(i,j)=4J_z\,|i-j|^{-\alpha}.$$

$U$ is diagonal in the **configuration basis** $|c\rangle$, where $c$ lists the occupied sites:

$$E(c)=\sum_{i\in c}V(i)+\sum_{i<j\in c}W(i,j).\tag{1}$$

$T$ is purely off-diagonal: it connects $c$ to configurations one hop away, with matrix element exactly $J_x$.

Two facts we will use throughout:

- **Scaling.** $V,W\propto J_z$, so $E(c)=J_z\,\varepsilon(c;N,\alpha)$. Changing $J_z$ rescales every gap but does not change which encoding is best.
- **Mirror symmetry.** $V(N-1-i)=V(i)$ and $W$ depends only on $|i-j|$, so a configuration and its mirror image have the same energy.

### 1a. $V$ and $W$ as arrays

`W` is an $N\times N$ matrix. Setting the diagonal distance to infinity makes $|i-i|^{-\alpha}=0$ automatically. Then $V(i)$ is just $-\tfrac12$ times the row sum of `W`, since $-\tfrac12\cdot4J_z=-2J_z$.

get_V_W precomputes a table of coefficients once, and config_energy reads off the entries for whichever sites are occupied.

In [33]:
def get_V_W(N, alpha, Jz):
    """
    Precompute the table of coefficients for the confining potential V and the pair repulsion W:

        V(i)   = -2 Jz * sum_{j != i} |i-j|^(-alpha)      (array of length N)
        W(i,j) =  4 Jz * |i-j|^(-alpha)                   (N x N matrix, W[i,i] = 0)
    """
    # np.arange(N) is the array [0, 1, ..., N-1] of site labels.
    #
    # np.subtract.outer(a, b) makes the matrix of all differences a[i] - b[j].
    # For N = 4:
    #     [[ 0 -1 -2 -3]
    #      [ 1  0 -1 -2]
    #      [ 2  1  0 -1]
    #      [ 3  2  1  0]]
    # np.abs makes it the distance matrix r[i, j] = |i - j|.
    # .astype(float) converts integers to floats, so we can put infinity in it next.
    r = np.abs(np.subtract.outer(np.arange(N), np.arange(N))).astype(float)

    # The diagonal is r[i, i] = 0, and 0 ** (-alpha) would be infinite.
    # A site doesn't interact with itself, so we want W[i, i] = 0 instead.
    # Setting the diagonal distance to infinity does exactly that: inf ** (-alpha) = 0.
    np.fill_diagonal(r, np.inf)

    # Elementwise power: every entry of r is raised to -alpha, then multiplied by 4 Jz.

    W = 4.0 * Jz * r ** (-alpha)

    # On-site confining potential V(i):
    # -0.5 * W.sum(axis=1) equals -2 * Jz * sum_{j != i} |i-j|^(-alpha)
    V = -0.5 * W.sum(axis=1)

    return V, W


def config_energy(occ, V, W):
    """
    Diagonal energy of a configuration,

        E(c) = sum_{i in c} V(i)  +  sum_{i < j in c} W(i, j)

    occ : list of occupied sites, e.g. [4, 5, 6]

    will return V[4]+V[5]+V[6]+W[4,5]+W[4,6]+W[5,6]
    """
    # Turn the list into a numpy array so it can be used to index V and W.
    occ = np.asarray(occ)

    # V[occ] picks out the entries of V at the occupied sites and .sum() adds them: 
    # this is the potential-energy term.
    potential = V[occ].sum()

    # np.ix_(occ, occ) selects the rows AND columns in occ, i.e. the sub-matrix
    # of W restricted to occupied sites. For occ = [4, 5]:
    #     [[W(4,4) W(4,5)]      [[0. 4.]
    #      [W(5,4) W(5,5)]]  =   [4. 0.]]
    # Summing it gives W(4,5) + W(5,4) = 2 W(4,5): every pair is counted twice
    # (once as (i,j), once as (j,i)), and the diagonal is 0.
    # The factor 0.5 removes the double counting -> sum over i < j only.
    repulsion = 0.5 * W[np.ix_(occ, occ)].sum()

    return potential + repulsion


# ---------------- example ----------------

# V, W = get_V_W(6, 0, 1.0)              # N = 6 sites, alpha = 0.5, Jz = 1
# print("V =", np.round(V, 4))             # np.round(..., 4): show 4 decimals

# # Configuration 000011: excitations on sites 4 and 5.
# #   V(4) + V(5) + W(4,5) = -7.5689 + (-6.4633) + 4.0 = -10.0323
# print("E([4,5]) =", round(config_energy([4, 5], V, W), 4))

**What to see:** $V$ is least negative at sites 0 and 5. The edges are the **maxima** of the convex well. Moving toward the center lowers the energy.


### 1b. Check against QuTiP

`config_energy` is a hand-written shortcut, so we check it against the full operator. We build $H_{\rm eff}$ in QuTiP on all $2^N$ states, keep the $b$-excitation block, and compare its diagonal with Eq. (1) for every configuration.

`site_op` puts a single-site operator on site $i$ and identities elsewhere: $\mathbb1\otimes\cdots\otimes\hat O\otimes\cdots\otimes\mathbb1$. Site 0 is the leftmost bit of the basis index.

In [38]:
def site_op(op, i, N):
    ops = [qt.qeye(2)] * N
    ops[i] = op
    return qt.tensor(ops)


def build_Heff_qutip(N, alpha, Jz, Jx):
    """
    H_eff = T + U built WITHOUT V and W:

        T = Jx * sum_i (a_i^dag a_{i+1} + h.c.)                  hopping
        U = sum_{i<j} Jz/|i-j|^alpha * sz_i sz_j  -  E0          spin Ising term minus constant
    """
    a  = [site_op(qt.destroy(2), i, N) for i in range(N)]   # a_i: removes an excitation at i
    sz = [site_op(qt.sigmaz(), i, N) for i in range(N)]     # sigma^z_i: +1 empty, -1 occupied

    H = 0

    # T: nearest-neighbour hopping, open chain (bonds 0-1, 1-2, ..., (N-2)-(N-1))
    for i in range(N - 1):
        H += Jx * (a[i].dag() * a[i + 1] + a[i + 1].dag() * a[i])

    # U: the original long-range Ising term, accumulating the constant E0 as we go
    E0 = 0.0
    for i in range(N):
        for j in range(i + 1, N):                            # each pair once (j > i)
            Jij = Jz / abs(i - j) ** alpha                   # coupling Jz / |i-j|^alpha
            H  += Jij * sz[i] * sz[j]
            E0 += Jij                                        # E0 = sum_{i<j} Jz / |i-j|^alpha

    # subtract E0 times the identity, so U matches sum V n + sum W n n with no offset
    # (qt.qeye([2]*N) is the identity on N spins, with the same dims as H)
    H -= E0 * qt.qeye([2] * N)

    return H


def occupied(k, N):
    # basis index k -> list of occupied sites, e.g. 3 -> '000011' -> [4, 5]
    return [i for i, bit in enumerate(format(k, f"0{N}b")) if bit == "1"]


N, alpha, Jz, Jx, b = 3, 0.5, 1.0, 0.1, 2
H = build_Heff_qutip(N, alpha, Jz, Jx)
Ntot = sum(site_op(qt.num(2), i, N) for i in range(N))

idx = [k for k in range(2**N) if bin(k).count("1") == b]     # basis states with b excitations
Hb = H.full()[np.ix_(idx, idx)]                              # the sector-b block


V, W = get_V_W(N, alpha, Jz)
E_fast = np.array([config_energy(occupied(k, N), V, W) for k in idx])

# print("Hermitian:              ", H.isherm)
# print("[H, Ntot] = 0:          ", (H * Ntot - Ntot * H).norm())
# print("[H, R] = 0:             ", (H.permute(list(range(N))[::-1]) - H).norm())
print("sector dimension:       ", Hb.shape[0], "=", comb(N, b))
print("b projected Hamiltonian", Hb)
print("max |E_fast - diag(H)|: ", np.abs(E_fast - np.diag(Hb).real).max())# is within numerical precision

sector dimension:        3 = 3
b projected Hamiltonian [[-3.41421356+0.j  0.1       +0.j  0.        +0.j]
 [ 0.1       +0.j -4.        +0.j  0.1       +0.j]
 [ 0.        +0.j  0.1       +0.j -3.41421356+0.j]]
max |E_fast - diag(H)|:  4.440892098500626e-16


**Expected:** `True`, `0.0`, `0.0`, `15 = 15`, and an error around $10^{-15}$.

- $[H,\hat N_{\rm tot}]=0$: hopping moves excitations but never creates them, so $b$ is conserved.
- $[H,R]=0$: `.permute(reversed order)` mirrors the chain. Reflection is an exact symmetry.
- The last line says Eq. (1) is exactly the diagonal of $H_{\rm eff}$. From here on we use `config_energy` and never build the $2^N$ matrix again.

## 2. The codespace as a list

**Pair assignment.** $t=(t_0,\dots,t_{N/2-1})$ with $t_p\in\{0,1,2\}$ and $\sum_pt_p=b$. The **active pairs** are $A=\{p:t_p=1\}$, with $|A|=2K$.

**Codewords.** Each active pair holds one excitation, on its left or its right site. So a configuration is fully specified by $S\subseteq A$, the active pairs whose excitation is on the **left**.

- $M=0$ forces exactly $K$ left-occupied pairs: $|S|=K$.
- Reflection maps $S\to S^c=A\setminus S$. The $R$-even codeword is

$$|v_S\rangle=\tfrac1{\sqrt2}\big(|S\rangle+|S^c\rangle\big),\qquad |v_S\rangle=|v_{S^c}\rangle,\qquad \dim\mathcal C=\tfrac12\binom{2K}{K}.$$

Because $S$ and $S^c$ give the same codeword, we keep only one of them: the one containing the first active pair.

Since $E(S)=E(S^c)$ by mirror symmetry, every codeword is an exact eigenstate of $U$. Without hopping, codewords would be frozen forever.

In [68]:
import itertools
from collections.abc import Set
from math import comb
import numpy as np


def config_from_S(S: Set[int], t: tuple[int, ...], N: int) -> list[int]:
    """
    Turn a codeword label S into the list of occupied sites of |S>.

    Args:
        S : active pairs whose excitation sits on the LEFT site, e.g. frozenset({0, 2})
        t : pair assignment, t[p] in {0, 1, 2} = excitations in pair p, e.g. (1,1,1,1,0,0)
        N : number of sites, with len(t) == N // 2, e.g. 12

    Returns:
        sorted list of occupied sites, e.g. [0, 2, 8, 10]

    Example (N = 12, t = (1,1,1,1,0,0), S = {0, 2}):
        pair 0 active, in S      -> left site 0
        pair 1 active, not in S  -> right site 10
        pair 2 active, in S      -> left site 2
        pair 3 active, not in S  -> right site 8
        pairs 4, 5 empty         -> nothing
        result: [0, 2, 8, 10]    i.e.  101000001010
    """
    occ = []                                     # occupied sites, filled pair by pair

    # enumerate(t) gives (index, value) pairs: (0, t[0]), (1, t[1]), ...
    # so p is the pair number and tp is how many excitations pair p holds
    for p, tp in enumerate(t):

        # the two sites of mirror pair p: site p on the left, its mirror N-1-p on the right
        L, R = p, N - 1 - p

        if tp == 2:
            # doubly occupied pair: both sites are occupied, no choice to make
            occ += [L, R]                        # += on a list appends both elements

        elif tp == 1:
            # active pair: exactly one excitation, and S decides which side
            # "L if p in S else R" is Python's one-line if/else:
            #   p in S     -> excitation on the left site
            #   p not in S -> excitation on the right site
            occ.append(L if p in S else R)

        # tp == 0: empty pair, nothing to add

    # sort so the same configuration always gives the same list
    # (needed later when we look configurations up or compare them)
    return sorted(occ)


def codewords(t: tuple[int, ...]) -> tuple[list[frozenset[int]], list[int], int]:
    """
    List all codewords |v_S> for pair assignment t.

    Args:
        t : pair assignment, t[p] in {0, 1, 2}, e.g. (1,1,1,1,0,0)

    Returns:
        words : one label S per codeword, e.g. [frozenset({0,1}), frozenset({0,2}), frozenset({0,3})]
        A     : active pairs (t[p] == 1), e.g. [0, 1, 2, 3]
        K     : number of left-occupied active pairs, |S| = K, e.g. 2
    """
    # active pairs: every pair p holding exactly one excitation
    # (list comprehension: "keep p for each (p, tp) in enumerate(t) if tp == 1")
    A = [p for p, tp in enumerate(t) if tp == 1]

    # M = 0 needs equal numbers of left and right excitations among active pairs,
    # so exactly half of them are left-occupied.  // is integer division.
    K = len(A) // 2

    # itertools.combinations(A, K) yields every way to pick K pairs out of A,
    # e.g. combinations([0,1,2,3], 2) -> (0,1) (0,2) (0,3) (1,2) (1,3) (2,3).
    #
    # S and its complement A \ S give the SAME codeword, because
    # |v_S> = (|S> + |S^c>)/sqrt(2) = |v_{S^c}>.  Of each such pair exactly one
    # contains the first active pair A[0], so "if A[0] in S" keeps one of each:
    #   kept:    (0,1) (0,2) (0,3)
    #   dropped: (2,3) (1,3) (1,2)   <- complements of the kept ones
    #
    # frozenset turns the tuple into an (immutable) set, so we can later use
    # set operations like "p in S", A - S, S & T, S ^ T.
    words = [frozenset(S) for S in itertools.combinations(A, K) if A[0] in S]

    return words, A, K


# ---------------- example ----------------

N, alpha, Jz = 12, 0.5, 1.0
t = (1, 1, 1, 1, 0, 0)                 # pairs 0-3 active (edges), pairs 4-5 empty (bulk)
words, A, K = codewords(t)             # unpack the three returned values
V, W = get_V_W(N, alpha, Jz)

# check the number of codewords against dim C = (1/2) * binomial(2K, K)
# (inside an f-string, {...} is replaced by the value of the expression)
print(f"A = {A}, K = {K}, dim C = {len(words)} (formula: {comb(2*K, K)//2})\n")

for S in words:
    # occupied sites of |S> and of its mirror image |S^c>
    occS  = config_from_S(S, t, N)
    occSc = config_from_S(frozenset(A) - S, t, N)      # A - S on sets = A \ S

    # bitstring picture: '1' on occupied sites, '0' elsewhere, site 0 on the left
    # ("".join glues the characters together into one string)
    bits  = "".join("1" if i in occS else "0" for i in range(N))

    # imbalance M = (# excitations in right half) - (# in left half)
    # "i >= N // 2" is True/False; sum() counts the Trues
    M     = sum(i >= N // 2 for i in occS) - sum(i < N // 2 for i in occS)

    # E(S) and E(S^c) should be equal by mirror symmetry
    # (:.4f in an f-string prints the number with 4 decimals)
    print(sorted(S), bits, f"M = {M}",
          f"E(S) = {config_energy(occS, V, W):.4f}", f"E(S^c) = {config_energy(occSc, V, W):.4f}")

A = [0, 1, 2, 3], K = 2, dim C = 3 (formula: 3)

[0, 1] 110000001100 M = 0 E(S) = -35.1489 E(S^c) = -35.1489
[0, 2] 101000001010 M = 0 E(S) = -37.4394 E(S^c) = -37.4394
[0, 3] 100100000110 M = 0 E(S) = -36.7701 E(S^c) = -36.7701


**What to see:** one excitation per active pair, two on each half, nothing in the bulk; $M=0$ in every row; $E(S)=E(S^c)$ in every row.

**Listing all pair assignments.** The search must not assume edge placement, so we generate every assignment with $\sum_pt_p=b$ and at least two codewords ($K\ge2$). With $d$ doubly occupied pairs there are $a=b-2d$ active pairs, giving $\binom{m}{d}\binom{m-d}{a}$ assignments for $m=N/2$ pairs.

In [ ]:
def pair_assignments(N: int, b: int, K_min: int = 2):
    """
    Generate every pair assignment t for b excitations on N sites.

    A pair assignment says how many excitations each mirror pair p = (p, N-1-p) holds:
        t[p] = 0  empty pair
        t[p] = 1  active pair    (one excitation, left or right: this is where the code lives)
        t[p] = 2  doubly occupied (both sites occupied)

    Constraints imposed:
        sum(t) == b                  correct excitation sector
        number of active pairs a     must be even (M = 0 needs a/2 left, a/2 right)
        a >= 2 * K_min               at least K_min left-occupied pairs; K_min = 2
                                     guarantees >= 3 codewords, so a logical pair (S, T) exists
        a + d <= N/2                 active + doubly occupied pairs fit in the N/2 pairs available

    No assumption about WHERE the pairs sit (edges, bulk, gaps): every placement is generated,
    so that "edge placement is optimal" is a result of the search, not an input.

    Args:
        N     : number of sites (even), e.g. 12
        b     : number of excitations, e.g. 6
        K_min : minimum number of left-occupied active pairs (default 2)

    Yields:
        t : tuple of length N/2 with entries in {0, 1, 2}, one assignment at a time

    Example (N = 12, b = 6 -> 6 pairs, 31 assignments):
        d = 0 doubles, a = 6 actives:  (1, 1, 1, 1, 1, 1)                     1 assignment
        d = 1 double,  a = 4 actives:  (2, 1, 1, 1, 1, 0), (2, 1, 1, 1, 0, 1), ...   30 assignments
        d = 2 doubles, a = 2 actives:  skipped, a < 2 * K_min
        d = 3 doubles, a = 0 actives:  skipped, no code

    Count: for each allowed d, choose which d pairs are doubly occupied, then which a of the
    remaining pairs are active:  sum_d  C(N/2, d) * C(N/2 - d, a).
    """
    m = N // 2                                   # number of mirror pairs

    # Each doubly occupied pair uses 2 excitations, each active pair uses 1,
    # so with d doubles there are a = b - 2d actives. Loop over every possible d.
    for d in range(b // 2 + 1):
        a = b - 2 * d

        # skip impossible or useless cases:
        #   a < 2*K_min : too few active pairs for two distinct codewords
        #   a % 2       : odd number of active pairs, can't have M = 0
        #   a + d > m   : more occupied pairs than pairs available
        if a < 2 * K_min or a % 2 or a + d > m:
            continue

        # choose which d pairs are doubly occupied
        # (combinations(range(5), 1) -> (0,) (1,) (2,) (3,) (4,))
        for doubles in itertools.combinations(range(m), d):

            # active pairs must come from the pairs that are not doubly occupied
            rest = [p for p in range(m) if p not in doubles]

            # choose which a of the remaining pairs are active
            for actives in itertools.combinations(rest, a):

                # build t: start all-empty, then fill in doubles and actives
                t = [0] * m
                for p in doubles:
                    t[p] = 2
                for p in actives:
                    t[p] = 1

                # "yield" hands back one t and pauses here until the next one is asked for.
                # This makes the function a generator: it never stores all assignments
                # in memory at once (230,300 of them at N = 100, b = 4).
                yield tuple(t)


# Count assignments without storing them:
# "sum(1 for _ in gen)" adds 1 for every t the generator produces.
# ("_" is the conventional name for a loop variable we don't use.)
# {N_:3d} pads the number to width 3, {...:7d} to width 7, so the columns line up.
for N_, b_ in [(20, 4), (20, 8), (60, 4), (100, 4)]:
    print(f"N = {N_:3d}, b = {b_}: {sum(1 for _ in pair_assignments(N_, b_)):7d} assignments")

N =  20, b = 4:     210 assignments
N =  20, b = 8:    4035 assignments
N =  60, b = 4:   27405 assignments
N = 100, b = 4:  230300 assignments


## 3. Leakage from a codeword

### 3a. Where the formula comes from

Take a configuration $|c\rangle$ with hop neighbors $|k\rangle$, $\langle k|T|c\rangle=J_x$, and gaps $\Delta_k=E(k)-E(c)$. If every hop is off-resonant, $|\Delta_k|\gg J_x$, first-order perturbation theory gives the eigenstate connected to $|c\rangle$:

$$|\tilde n_c\rangle=|c\rangle-\sum_k\frac{J_x}{\Delta_k}|k\rangle+\dots,\qquad |\langle c|\tilde n_c\rangle|^2=1-\sum_k\Big(\frac{J_x}{\Delta_k}\Big)^2+O(J_x^4/\Delta^4).$$

The long-time-averaged return probability is $\overline P(c)=\sum_n|\langle n|c\rangle|^4$. The $\tilde n_c$ term gives $1-2\sum_k(J_x/\Delta_k)^2$, and all others are $O(J_x^4)$. So

$$\boxed{L(c)=1-\overline P(c)=2\sum_{k}\Big(\frac{J_x}{\Delta_k}\Big)^2}\tag{2}$$

This is the paper's two-level lemma, $2(\lambda/\Delta)^2$ per channel, summed over channels. For $K\ge2$ the neighbors of $S$ and $S^c$ don't overlap, so $L(v_S)=L(S)$.

**Why the sum and not the smallest gap:** $2(J_x/\Delta_{\min})^2\le L\le|\mathcal N|\cdot2(J_x/\Delta_{\min})^2$, where $|\mathcal N|\le2b$ is the number of hops. A codeword with many moderately soft hops can leak more than one with a single softer hop. §6 shows this happening.

### 3b. The hop gap

When one excitation hops $i\to j=i\pm1$, only the terms involving it change:

$$\Delta_{i\to j}(c)=\underbrace{V(j)-V(i)}_{\text{Stark step}}+\underbrace{\sum_{k\in c,\,k\ne i}\big[W(j,k)-W(i,k)\big]}_{\text{change in repulsion}}.\tag{3}$$

The Stark step pulls excitations toward the center (negative in the left half). The repulsion term pushes them away from each other (positive when moving toward a neighbor). When the two nearly cancel, the hop is resonant even for $J_x\ll J_z$.

A hop is allowed only if $j$ is inside the chain and empty (hard-core bosons).

In [45]:
def hop_gaps(occ, V, W, N):
    # all allowed single hops (i, j, Delta), Eq. (3)
    occ_set, occ_arr = set(occ), np.asarray(occ)
    out = []
    for i in occ:
        others = occ_arr[occ_arr != i]
        for j in (i - 1, i + 1):
            if 0 <= j < N and j not in occ_set:
                delta = V[j] - V[i] + (W[j, others] - W[i, others]).sum()
                out.append((i, j, delta))
    return out


def leakage(occ, V, W, N, Jx):
    # Eq. (2). Also returns the smallest |gap| to check the formula is valid.
    gaps = np.array([abs(g) for (_, _, g) in hop_gaps(occ, V, W, N)])
    if len(gaps) == 0:
        return 0.0, np.inf
    return 2.0 * np.sum((Jx / gaps) ** 2), gaps.min()


N, alpha, Jz, Jx = 20, 0.5, 100.0, 1.0
t = (1, 1, 1, 1) + (0,) * 6
words, A, K = codewords(t)
V, W = get_V_W(N, alpha, Jz)

for S in words:
    occ = config_from_S(S, t, N)
    L, gmin = leakage(occ, V, W, N, Jx)
    hops = [(int(i), int(j), round(float(g), 1)) for (i, j, g) in hop_gaps(occ, V, W, N)]
    print(f"S = {sorted(S)}  occ = {occ}")
    print(f"   hops (from, to, gap): {hops}")
    print(f"   L = {L:.3e}   smallest gap = {gmin:.1f} Jx\n")

S = [0, 1]  occ = [0, 1, 16, 17]
   hops (from, to, gap): [(1, 2, -204.5), (16, 15, -160.3), (17, 18, -28.6)]
   L = 2.572e-03   smallest gap = 28.6 Jx

S = [0, 2]  occ = [0, 2, 16, 18]
   hops (from, to, gap): [(0, 1, -30.9), (2, 1, 204.8), (2, 3, -111.6), (16, 15, -94.6), (16, 17, 177.5), (18, 17, 28.9), (18, 19, 96.7)]
   L = 5.194e-03   smallest gap = 28.9 Jx

S = [0, 3]  occ = [0, 3, 17, 18]
   hops (from, to, gap): [(0, 1, -96.5), (3, 2, 112.0), (3, 4, -73.3), (17, 16, -177.1), (18, 19, 31.2)]
   L = 2.870e-03   smallest gap = 31.2 Jx



**What to see:** $\{0,2\}$ does not have the smallest minimum gap, but it has 7 hops instead of 3 or 5, and it leaks the most. A smallest-gap ranking would get this wrong.

Also check the scaling: rerun with `Jz = 10.0`. Every gap shrinks by 10 and every $L$ grows by 100, but the **ranking** of codewords doesn't change. Only $N$, $\alpha$ and $t$ decide the ranking.

## 4. Choosing the logical pair

For $|0_L\rangle=|v_S\rangle$, $|1_L\rangle=|v_T\rangle$ we need three quantities.

**X-distance** (static Pauli-$X$ errors, from the paper):

$$d_X=2\min\big(|S\triangle T|,\;|S\triangle T^c|\big).$$

**Hop distance** $h(S,T)$: the minimum number of coherent hops turning configuration $S$ into $T$ or $T^c$. Hard-core bosons can't pass each other in 1D, so the optimal path matches the sorted positions:

$$h=\sum_k|x_k-y_k|,\qquad x,y\ \text{sorted occupied sites}.$$

This is **not** $d_X$. Hopping can move an excitation to a *neighboring* active pair in one step, so two codewords at $d_X=4$ can be only 2 hops apart. At order $h$ the hopping generates a direct logical coupling $J_{\rm eff}\propto J_x^h/J_z^{h-1}$, and the time-averaged logical transfer is

$$\overline P_{S\to T}\approx2\Big(\frac{J_{\rm eff}}{\Delta E_{ST}}\Big)^2\propto\Big(\frac{J_x}{J_z}\Big)^{2h}.\tag{4}$$

**Energy mismatch** $\Delta E_{ST}=E(S)-E(T)$ sets the logical phase rate. A drift $\delta J_z/J_z=\epsilon$ produces a phase error $\epsilon\,\Delta E_{ST}\,t$. Making $\Delta E_{ST}=0$ removes that, but then Eq. (4) breaks down and the qubit Rabi-flips at rate $J_{\rm eff}$. **So degeneracy is only safe when $h$ is large.**

In [44]:
def d_X(S, T, A):
    Tc = frozenset(A) - T
    return 2 * min(len(S ^ T), len(S ^ Tc))          # ^ = symmetric difference of sets


def hop_distance(occ1, occ2):
    return int(np.abs(np.asarray(sorted(occ1)) - np.asarray(sorted(occ2))).sum())


def h_ST(S, T, t, A, N):
    occS  = config_from_S(S, t, N)
    occT  = config_from_S(T, t, N)
    occTc = config_from_S(frozenset(A) - T, t, N)
    return min(hop_distance(occS, occT), hop_distance(occS, occTc))


for S, T in itertools.combinations(words, 2):
    ES = config_energy(config_from_S(S, t, N), V, W)
    ET = config_energy(config_from_S(T, t, N), V, W)
    print(f"S = {sorted(S)}, T = {sorted(T)}:  d_X = {d_X(S, T, A)},  h = {h_ST(S, T, t, A, N)},"
          f"  |dE| = {abs(ES - ET) / Jx:.1f} Jx")

S = [0, 1], T = [0, 2]:  d_X = 4,  h = 2,  |dE| = 233.4 Jx
S = [0, 1], T = [0, 3]:  d_X = 4,  h = 4,  |dE| = 167.9 Jx
S = [0, 2], T = [0, 3]:  d_X = 4,  h = 2,  |dE| = 65.5 Jx


**What to see:** all three pairs have $d_X=4$, but $h$ is 2 for two of them and 4 for $\{0,1\},\{0,3\}$. By Eq. (4), the $h=4$ pair is protected against coherent logical mixing by an extra factor of roughly $(J_x/J_z)^4$. §6 confirms this.

## 5. The optimization

For every assignment $t$, score each codeword **once**, then rank all pairs $(S,T)$ by

1. smallest $\max\big(L(S),L(T)\big)$: the qubit is only as good as its worse codeword,
2. then largest $h(S,T)$,
3. then smallest $|\Delta E_{ST}|$.

Optional constraints: minimum $d_X$ and minimum $h$.

Each row records the smallest gap of its two codewords (`gap_min`, in units of $J_x$). If it's below about $5J_x$, Eq. (2) isn't reliable and the row is marked `valid = False`.

In [46]:
def best_pairs_for_assignment(t, N, V, W, Jx, dX_min=0, h_min=0):
    words, A, K = codewords(t)
    if len(words) < 2:
        return []
    info = {}                                         # score each codeword once
    for S in words:
        occ = config_from_S(S, t, N)
        L, gmin = leakage(occ, V, W, N, Jx)
        info[S] = (L, gmin, config_energy(occ, V, W))

    rows = []
    for S, T in itertools.combinations(words, 2):
        dx = d_X(S, T, A)
        if dx < dX_min:
            continue
        h = h_ST(S, T, t, A, N)
        if h < h_min:
            continue
        (LS, gS, ES), (LT, gT, ET) = info[S], info[T]
        rows.append(dict(t="".join(map(str, t)), K=K, S=tuple(sorted(S)), T=tuple(sorted(T)),
                         dX=dx, h=h, L_pair=max(LS, LT),
                         gap_min=min(gS, gT) / Jx, dE=abs(ES - ET) / Jx))
    return rows


def optimize(N, b, alpha, Jz, Jx, dX_min=0, h_min=0, gap_valid=5.0, top=10):
    V, W = get_V_W(N, alpha, Jz)
    rows = []
    for t in pair_assignments(N, b):
        rows += best_pairs_for_assignment(t, N, V, W, Jx, dX_min, h_min)
    rows.sort(key=lambda r: (r["L_pair"], -r["h"], r["dE"]))
    for r in rows:
        r["valid"] = r["gap_min"] >= gap_valid
    return rows[:top] if top else rows


def show(rows):
    print(f"{'#':>3} {'t':<12} {'K':>2} {'S':<10} {'T':<10} {'dX':>3} {'h':>3} {'L_pair':>10} {'gap_min':>8} {'|dE|':>8} valid")
    for k, r in enumerate(rows, 1):
        print(f"{k:>3} {r['t']:<12} {r['K']:>2} {str(r['S']):<10} {str(r['T']):<10} {r['dX']:>3} {r['h']:>3} "
              f"{r['L_pair']:>10.3e} {r['gap_min']:>8.1f} {r['dE']:>8.1f} {r['valid']}")

In [49]:
N, b, alpha, Jz, Jx = 20, 4, 0.4, 100.0, 1.0
best = optimize(N, b, alpha, Jz, Jx, top=10)
show(best)

  # t             K S          T           dX   h     L_pair  gap_min     |dE| valid
  1 1110100000    2 (0, 1)     (0, 4)       4   4  3.288e-03     35.1     71.5 True
  2 1101100000    2 (0, 1)     (0, 3)       4   4  3.476e-03     36.5    281.7 True
  3 1011100000    2 (0, 2)     (0, 3)       4   2  4.142e-03     34.3    140.7 True
  4 1110100000    2 (0, 2)     (0, 4)       4   2  4.440e-03     34.9     69.5 True
  5 1110100000    2 (0, 1)     (0, 2)       4   2  4.440e-03     34.9    141.0 True
  6 1100110000    2 (0, 1)     (0, 4)       4   6  4.571e-03     36.7    334.2 True
  7 1001110000    2 (0, 3)     (0, 5)       4   4  4.665e-03     30.0     43.7 True
  8 1110010000    2 (0, 2)     (0, 5)       4   2  4.763e-03     32.9     78.3 True
  9 1100000011    2 (0, 1)     (0, 8)       4  14  5.177e-03     28.8    342.1 True
 10 0000001111    2 (6, 7)     (6, 8)       4   2  5.469e-03     23.5    163.1 True


**How to read `t`:** one digit per mirror pair, starting from the outermost pair. `0` = empty, `1` = active, `2` = doubly occupied. So `1110100000` means pairs 0, 1, 2 and 4 are active and pair 3 is skipped.

**What to see:** the best assignment is edge-heavy but **not** the four outermost pairs (`1111000000`). Skipping a pair spreads the excitations out and weakens the repulsion term in Eq. (3) that softens the hops.

## 6. Exact verification

Eq. (2) is perturbative, so we check the top candidates against exact dynamics. We build $H^b_{\rm eff}$ as a dense matrix in the configuration basis of sector $b$ (dimension $\binom Nb$), diagonalize it, and compute time averages with

$$\overline{|\langle\phi|e^{-iHt}|\psi\rangle|^2}=\sum_E\big|\langle\phi|P_E|\psi\rangle\big|^2,$$

grouping degenerate eigenvalues ($P_E$ is the projector onto the eigenspace with energy $E$). The grouping matters because the spectrum has exact mirror degeneracies.

This is the same matrix as QuTiP's $H_{\rm eff}$ in §1b, built directly in the sector so that $N=20$ fits in memory.

In [50]:
def build_Heff_sector(N, b, alpha, Jz, Jx):
    V, W = get_V_W(N, alpha, Jz)
    basis = list(itertools.combinations(range(N), b))
    index = {c: k for k, c in enumerate(basis)}
    H = np.zeros((len(basis), len(basis)))
    for k, c in enumerate(basis):
        H[k, k] = config_energy(list(c), V, W)
        occ = set(c)
        for i in c:
            j = i + 1                                 # each bond once; add both directions
            if j < N and j not in occ:
                new = tuple(sorted((occ - {i}) | {j}))
                H[index[new], k] += Jx
                H[k, index[new]] += Jx
    return H, index


def time_averaged(e, U, psi, phi, tol=1e-9):
    a, c = U.T @ psi, U.T @ phi
    total, k = 0.0, 0
    while k < len(e):
        m = k
        while m + 1 < len(e) and abs(e[m + 1] - e[k]) < tol:
            m += 1
        total += np.dot(c[k:m + 1], a[k:m + 1]) ** 2
        k = m + 1
    return total


def verify(row, N, alpha, Jz, Jx):
    t = tuple(int(x) for x in row["t"])
    S, T = frozenset(row["S"]), frozenset(row["T"])
    b = sum(t)
    A = [p for p, tp in enumerate(t) if tp == 1]
    H, index = build_Heff_sector(N, b, alpha, Jz, Jx)
    e, U = np.linalg.eigh(H)

    def vec(X):
        v = np.zeros(len(e))
        v[index[tuple(config_from_S(X, t, N))]] += 1 / np.sqrt(2)
        v[index[tuple(config_from_S(frozenset(A) - X, t, N))]] += 1 / np.sqrt(2)
        return v

    V, W = get_V_W(N, alpha, Jz)
    vS, vT = vec(S), vec(T)
    for name, X, v in (("S", S, vS), ("T", T, vT)):
        L_ed = 1 - time_averaged(e, U, v, v)
        L_pt = leakage(config_from_S(X, t, N), V, W, N, Jx)[0]
        print(f"  {name} = {tuple(sorted(X))}:  L_ED = {L_ed:.3e}   L_Eq2 = {L_pt:.3e}   ratio = {L_ed / L_pt:.3f}")
    print(f"  logical transfer S -> T (ED): {time_averaged(e, U, vS, vT):.3e}")

In [51]:
for r in best[:2]:
    print(f"t = {r['t']}, S = {r['S']}, T = {r['T']}, h = {r['h']}")
    verify(r, N, alpha, Jz, Jx)
    print()

t = 1110100000, S = (0, 1), T = (0, 4), h = 4
  S = (0, 1):  L_ED = 3.273e-03   L_Eq2 = 3.288e-03   ratio = 0.995
  T = (0, 4):  L_ED = 3.170e-03   L_Eq2 = 3.179e-03   ratio = 0.997
  logical transfer S -> T (ED): 4.857e-15

t = 1101100000, S = (0, 1), T = (0, 3), h = 4
  S = (0, 1):  L_ED = 1.933e-03   L_Eq2 = 1.505e-03   ratio = 1.284
  T = (0, 3):  L_ED = 3.589e-03   L_Eq2 = 3.476e-03   ratio = 1.032
  logical transfer S -> T (ED): 1.041e-12



**What to see:**

- For the winner, the leakage formula matches ED to within about 1%, and the logical transfer is around $10^{-15}$ ($h=4$).
- For the runner-up, one codeword is off by about 30%. Eq. (2) is a good *ranking* tool, but quote ED numbers in the paper, and always verify the top few candidates.

ED costs about 15 s per candidate at $N=20$, $b=4$ ($\binom{20}{4}=4845$ states). For larger sectors, run it in the $R$-even block or use sparse time evolution.

## 7. Sweep over $\alpha$

The ranking depends on $\alpha$ and $N$, so $\alpha$ should be a search variable rather than a fixed input. Remember the constraint from Requirement 3: sector leakage scales as $N^{2\alpha-1}$, so only $\alpha<1/2$ is admissible.

This sweep reports only the **intra-sector** part. The total to minimize is

$$L_{\rm tot}=L_{\rm sector}(N,\alpha,b)+\max\big(L(S),L(T)\big),$$

and $L_{\rm sector}$ still needs its prefactor.

In [12]:
N, b, Jz, Jx = 40, 4, 100.0, 1.0
print(f"N = {N}, b = {b}, Jz/Jx = {Jz/Jx:g}\n")
print(f"{'alpha':>6} {'best L_pair':>12} {'t':<22} {'S':<10} {'T':<10} {'h':>3} valid")
for alpha in [0.1, 0.2, 0.3, 0.35, 0.4, 0.45]:
    r = optimize(N, b, alpha, Jz, Jx, top=1)[0]
    print(f"{alpha:>6.2f} {r['L_pair']:>12.3e} {r['t']:<22} {str(r['S']):<10} {str(r['T']):<10} {r['h']:>3} {r['valid']}")

N = 40, b = 4, Jz/Jx = 100

 alpha  best L_pair t                      S          T            h valid


  0.10    6.311e-03 11110000000000000000   (0, 1)     (0, 3)       4 True


  0.20    3.165e-03 11101000000000000000   (0, 2)     (0, 4)       2 True


  0.30    2.240e-03 11101000000000000000   (0, 2)     (0, 4)       2 True


  0.35    2.088e-03 11101000000000000000   (0, 2)     (0, 4)       2 True


  0.40    2.039e-03 11101000000000000000   (0, 2)     (0, 4)       2 True


  0.45    1.890e-03 11001100000000000000   (0, 1)     (0, 4)       6 True


**What to see:**

- The winning assignment changes with $\alpha$. At small $\alpha$ the four outermost pairs win; at intermediate $\alpha$ a pair is skipped.
- Several winners have $h=2$, because the ranking puts leakage first. If coherent logical mixing matters for your memory time, rerun with `optimize(..., h_min=4)` and compare.
- These are intra-sector numbers only. Smaller $\alpha$ improves sector leakage ($N^{2\alpha-1}$), so the full trade-off needs $L_{\rm sector}$.

## 8. Summary and what's next

**What the notebook establishes**

- Eq. (1) is exactly the diagonal of $H_{\rm eff}$ (§1b), so the scoring needs no large matrices.
- The codespace enumeration reproduces $\dim\mathcal C=\tfrac12\binom{2K}{K}$ (§2).
- The leakage score must be the **sum** over hops, Eq. (2). The smallest gap alone misranks codewords (§3, §6).
- Coherent logical mixing is controlled by the hop distance $h$, not by $d_X$ (§4, §6).
- The best assignment is edge-heavy but not necessarily the outermost contiguous pairs (§5).

**Next steps**

1. Fix the prefactor of $L_{\rm sector}$ and add it in an outer loop over $b$.
2. Run the sweep at the $N$ you'll report (e.g. $N=100$), and at larger $b$ to see whether $\alpha=0.4$ wins once more active pairs are needed.
3. Verify the final encoding by ED, or by sparse time evolution for sectors too large to diagonalize.